# Day 2 · 한국어 회의 Workflow와 Agent

하나의 입력을 8개 차시 동안 확장합니다. 웹사이트 확인이 아니라 코드·명령·test·결과 파일을 직접 다루며, 외부 서비스 저장·게시·발송은 dry-run과 사람 승인을 먼저 거칩니다.

In [ ]:
from pathlib import Path
import importlib.util, json, subprocess, sys

def find_workspace(start):
    for candidate in [start, *start.parents]:
        if (candidate / "requirements-day1.txt").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("WORKSPACE_ROOT_NOT_FOUND")

ROOT = find_workspace(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print({"workspace": ROOT.name, "python": sys.version.split()[0]})

In [ ]:
# Run All은 설치가 끝난 환경에서 network 호출 없이 실행합니다.
# 처음 한 번만 아래 flag를 True로 바꿔 현재 Notebook Kernel에 설치합니다.
INSTALL_CORE_DEPENDENCIES = False

dependency_groups = {
    "core": (["pydantic", "pytest", "langchain_core", "langgraph"], ROOT / "requirements-day1.txt"),
}
install_flags = {
    "core": INSTALL_CORE_DEPENDENCIES,
}
dependency_status = {}
for group, (modules, requirements_path) in dependency_groups.items():
    missing = [name for name in modules if importlib.util.find_spec(name) is None]
    if missing and install_flags[group]:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)],
            check=True,
        )
        missing = [name for name in modules if importlib.util.find_spec(name) is None]
    dependency_status[group] = {
        "ready": not missing,
        "missing": missing,
        "install_command": f"python -m pip install -r {requirements_path.relative_to(ROOT)}",
        "network_used_by_run_all": bool(install_flags[group]),
    }

assert dependency_status["core"]["ready"], (
    "CORE_DEPENDENCIES_MISSING: 위 INSTALL_CORE_DEPENDENCIES를 True로 바꾸고 이 셀만 먼저 실행하세요."
)
print(json.dumps(dependency_status, ensure_ascii=False, indent=2))

# faster-whisper model과 공개 음성 다운로드는 2차시 opt-in 셀에서만 실행합니다.

In [ ]:
OUT = ROOT / "output/course-labs/day2-v2"
OUT.mkdir(parents=True, exist_ok=True)

def save_json(name, payload):
    path = OUT / name
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print({"saved": str(path.relative_to(ROOT))})
    return path

def save_text(name, text):
    path = OUT / name
    path.write_text(text.rstrip() + "\n", encoding="utf-8")
    print({"saved": str(path.relative_to(ROOT))})
    return path

def run_command(*args, cwd=ROOT):
    completed = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    display_args = list(args)
    if display_args and display_args[0] == sys.executable:
        display_args[0] = "python"
    result = {
        "command": " ".join(display_args),
        "returncode": completed.returncode,
        "stdout_tail": completed.stdout.strip().splitlines()[-5:],
        "stderr_tail": completed.stderr.strip().splitlines()[-5:],
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result

## 1차시 · Meeting Agent Architecture

일반 사용자의 말로 시작해도 구현에서는 입력·맥락·변환·검증·승인·초안을 분리합니다. LLM은 한 단계의 판단 도구이고, Agent는 요청에 따라 정보원과 Workflow를 고르는 상위 실행 구조입니다.

In [ ]:
from src.course_services.day2_meeting_workflow import (
    DEFAULT_OPENAI_MODEL, DomainContext, MCPRetrievalPolicy, MeetingRecord,
    SourceInput, TranscriptEnvelope, build_mcp_retrieval_plan,
    build_interruptible_meeting_graph,
    compact_workflow_result, compare_execution_strategies,
    diagnose_provider_options, normalize_source, render_email_draft,
    resume_interruptible_meeting_review, route_execution_strategy,
    run_meeting_workflow, start_interruptible_meeting_review,
    run_optional_cli_prompt, run_optional_openai_prompt, run_optional_openai_record,
    source_mixing_error_example,
    validate_record_evidence,
)

architecture = {
    "user_request": "회의를 이해하고 근거 있는 기록·할 일·인사이트 초안을 만들어 줘",
    "layers": [
        {"order": 1, "name": "policy", "question": "읽어도 되는 정보와 하면 안 되는 행동은?"},
        {"order": 2, "name": "input_adapter", "question": "Meet·ClovaNote·음성 중 어떤 한 입력인가?"},
        {"order": 3, "name": "domain_context", "question": "산업 용어와 이전 결정은 무엇인가?"},
        {"order": 4, "name": "workflow", "question": "정규화→STT→구조화→근거 검증 순서는?"},
        {"order": 5, "name": "human_review", "question": "누가 승인·수정·거절하는가?"},
        {"order": 6, "name": "draft_export", "question": "MD·이메일 초안을 어디까지 만들 것인가?"},
    ],
    "two_meanings_of_agent": {
        "user_view": "여러 단계를 알아서 이어 주는 서비스",
        "engineering_view": "상태·도구·정책·오류·승인을 가진 실행 시스템",
    },
    "fixed_graph": [
        "policy", "input_normalize", "stt_optional", "structure",
        "evidence", "human_review", "export_draft",
    ],
    "invariants": {
        "source_count": 1,
        "human_review_required": True,
        "external_write": False,
        "run_all_network_calls": 0,
    },
}
assert architecture["invariants"]["external_write"] is False
save_json("01_architecture.json", architecture)
architecture

## 2차시 · Input Route · STT

Google Meet 텍스트, ClovaNote TXT, 로컬 음성은 출발점만 다릅니다. 텍스트가 이미 있으면 STT를 건너뛰고, 음성만 있을 때 로컬 STT를 실행합니다. 한 요청에 입력을 섞지 않고 모두 `TranscriptEnvelope`로 바꾼 뒤 같은 Workflow에 넣습니다.

In [ ]:
from src.meeting_demo import parse_transcript

meet_text = "\n".join([
    "[00:00] 민지: 오늘은 배송 지연 회의 기록 자동화 범위를 확정하겠습니다.",
    "[00:18] 준호: WISMO 문의를 우선 처리하고 환불 자동화는 보류하는 것이 좋겠습니다.",
    "[00:37] 서연: 제가 9월 2일까지 고객 안내 문구를 정리해 공유하겠습니다.",
    "[00:55] 민지: 최근 야근 부담이 있으니 이번 범위를 더 늘리지 않겠습니다.",
])
clova_text = "\n".join([
    "화자 1 00:00", "배송 지연 원인 분류를 1차 범위로 확정합니다.",
    "화자 2 00:24", "제가 9월 3일까지 분류 기준을 작성하겠습니다.",
    "화자 1 00:46", "운영팀 부담을 확인한 뒤 다음 범위를 결정하겠습니다.",
])
audio_path = ROOT / "data/meeting_sample_ko_12min.wav"

sources = {
    "google_meet_text": SourceInput(
        source_mode="google_meet_text", source_ref="meet://fixture/delivery-001",
        meet_transcript=meet_text,
        speaker_metadata={
            "민지": {"display_name": "민지", "role": "PM"},
            "준호": {"display_name": "준호", "role": "Engineer"},
            "서연": {"display_name": "서연", "role": "Operations"},
        },
        history_metadata={"prior_decisions": ["고객 자동 발송 금지"]},
    ),
    "clovanote_txt": SourceInput(
        source_mode="clovanote_txt", source_ref="clovanote-export-001.txt",
        clovanote_text=clova_text,
        speaker_metadata={
            "화자 1": {"display_name": "민지", "role": "PM"},
            "화자 2": {"display_name": "준호", "role": "Engineer"},
        },
    ),
    "audio_stt": SourceInput(
        source_mode="audio_stt", source_ref="synthetic-12min-audio",
        audio_path=str(audio_path),
    ),
}

def reviewed_fixture_stt(path):
    assert path.resolve() == audio_path.resolve()
    text = (ROOT / "data/meeting_sample_ko_12min.txt").read_text(encoding="utf-8")
    segments = parse_transcript(text)
    return text, segments, {
        "provider": "reviewed_fixture_stt", "language": "ko",
        "network_used": False, "matched_audio_transcript_pair": True,
    }

envelopes = {
    "google_meet_text": normalize_source(sources["google_meet_text"]),
    "clovanote_txt": normalize_source(sources["clovanote_txt"]),
    "audio_stt": normalize_source(
        sources["audio_stt"], transcriber=reviewed_fixture_stt
    ),
}
input_result = {
    "contracts": {
        name: {
            "source_mode": envelope.source_mode,
            "source_count": envelope.source_count,
            "segment_count": len(envelope.segments),
            "first_segment": envelope.segments[0].model_dump(mode="json"),
            "stt_metadata": envelope.stt_metadata,
        }
        for name, envelope in envelopes.items()
    },
    "boundary": source_mixing_error_example(),
}
assert {item["source_count"] for item in input_result["contracts"].values()} == {1}
assert input_result["boundary"]["error_code"] == "SOURCE_MODE_MIXING_FORBIDDEN"
save_json("02_inputs.json", input_result)
input_result

## 3차시 · Domain Context · MCP Policy

회의에서 말한 사실과 사용자가 제공한 업무 맥락을 분리합니다. Notion·Confluence·Slack이 필요해 보여도 자동으로 읽지 않고, 허용된 범위와 기간을 가진 MCP 읽기 계획만 먼저 만듭니다.

In [ ]:
domain_context = DomainContext(
    industry="이커머스 고객경험",
    organization_context="배송 지연 문의가 증가해 상담 부담과 고객 불편이 함께 커진 상태",
    meeting_objective="배송 지연 회의 기록 자동화 범위 확정",
    glossary={"WISMO": "배송 위치 문의", "SLA": "약속한 응답 시간"},
    prior_decisions=["외부 발송은 사람 승인 뒤에만 진행", "환불 자동화는 이번 범위에서 제외"],
    desired_outcomes=["근거가 있는 담당자별 To Do", "단기·중기·장기 인사이트"],
    confidentiality="internal",
)
retrieval_policy = MCPRetrievalPolicy(
    allowed_connectors=["notion", "confluence", "slack"],
    explicit_user_authorization=True,
    lookback_days=14,
    allowed_scopes={
        "notion": ["CX PoC"], "confluence": ["CX 정책"], "slack": ["#delivery-poc"],
    },
    participant_match_required=True,
    max_items_per_connector=5,
)
mcp_plan = build_mcp_retrieval_plan(
    envelope=envelopes["google_meet_text"],
    domain=domain_context,
    policy=retrieval_policy,
)
context_prompt_fields = {
    "industry": domain_context.industry,
    "organization_context": domain_context.organization_context,
    "meeting_objective": domain_context.meeting_objective,
    "glossary": domain_context.glossary,
    "prior_decisions": domain_context.prior_decisions,
    "desired_outputs": domain_context.desired_outcomes,
    "evidence_rule": "회의 발화의 사실 주장에는 s01 같은 evidence ID 필수",
}
context_result = {
    "domain_context": domain_context.model_dump(mode="json"),
    "context_prompt_fields": context_prompt_fields,
    "mcp_retrieval_plan": mcp_plan,
}
assert mcp_plan["executed"] is False and mcp_plan["external_write"] is False
save_json("03_domain_context.json", context_result)
context_result

## 4차시 · MeetingRecord Schema

먼저 모든 실행 방식이 반환해야 할 `MeetingRecord`를 고정합니다. 그 다음 한 번의 생성이면 단일 LLM, 고정 순서면 Workflow, 정보원과 다음 행동이 요청마다 달라지면 Agent Router를 선택합니다. 알려진 경로를 고르는 데 LLM을 쓰지 않으면 비용과 오작동 지점을 줄일 수 있습니다.

In [ ]:
record_contract_run = run_meeting_workflow(
    sources["google_meet_text"],
    domain_context,
    review_decision="approve",
    retrieval_policy=retrieval_policy,
)
actual_record = MeetingRecord.model_validate(record_contract_run["record"])
actual_envelope = TranscriptEnvelope.model_validate(record_contract_run["envelope"])
actual_schema = MeetingRecord.model_json_schema()
routing_cases = {
    "fixed_meeting_record": route_execution_strategy(
        requested_actions=["normalize", "summarize", "perspectives", "todos", "insights", "draft"]
    ),
    "context_retrieval_needed": route_execution_strategy(
        requested_actions=["summarize", "todos"],
        external_context_sources=["notion", "confluence", "slack"],
    ),
    "one_off_unmodeled_request": route_execution_strategy(
        requested_actions=["rewrite_as_podcast_script"]
    ),
}
record_contract_result = {
    "schema": {
        "name": "MeetingRecord",
        "field_names": list(MeetingRecord.model_fields),
        "required_fields": actual_schema.get("required", []),
        "properties": actual_schema["properties"],
    },
    "actual_record": actual_record.model_dump(mode="json"),
    "evidence_validation": {
        "known_segment_ids": [segment.id for segment in actual_envelope.segments],
        "errors": validate_record_evidence(actual_record, actual_envelope),
    },
    "execution_strategy_comparison": compare_execution_strategies(),
    "rule_router_examples": routing_cases,
    "delivery_policy": {
        "human_review_required": actual_record.human_review_required,
        "external_write": actual_record.external_write,
        "draft_status": record_contract_run["exports"]["status"],
    },
}
assert set(record_contract_result["actual_record"]) == set(MeetingRecord.model_fields)
assert record_contract_result["evidence_validation"]["errors"] == []
assert record_contract_result["delivery_policy"]["external_write"] is False
assert routing_cases["fixed_meeting_record"]["strategy"] == "deterministic_workflow"
assert routing_cases["context_retrieval_needed"]["strategy"] == "agent_router"
assert routing_cases["one_off_unmodeled_request"]["strategy"] == "single_llm"
save_json("04_meeting_record_contract.json", record_contract_result)
record_contract_result

## 5차시 · Coding Agent Workflow

Codex나 Claude Code에는 곧바로 “Agent를 만들어 줘”라고 하지 않습니다. 세 입력 시나리오, 공통 결과 계약, 금지 행동, 정상·실패 Test를 먼저 합의한 뒤 구현을 요청합니다. 아래 실행 결과가 대화형 코딩 Agent에게 전달할 인수 기준입니다.

In [ ]:
coding_agent_brief = {
    "scenarios": ["google_meet_text", "clovanote_txt", "audio_stt"],
    "implementation_request": (
        "세 입력을 하나의 MeetingRecord로 정규화하고, 근거 검증과 사람 승인 뒤 "
        "Markdown·이메일 초안까지만 만드는 코드를 구현해 주세요."
    ),
    "must_not": ["입력 자동 혼합", "근거 없는 담당자 추정", "승인 전 외부 저장·발송"],
    "acceptance_tests": [
        "세 입력 모두 같은 결과 계약", "존재하지 않는 evidence ID 차단",
        "승인·수정·거절 상태 분리", "모든 기본 실행에서 external_write=false",
    ],
    "conversation_guide": "materials/day2/Codex_Claude_대화_시나리오.md",
}
workflow_runs = {
    "google_meet_text": run_meeting_workflow(
        sources["google_meet_text"], domain_context,
        review_decision="approve", retrieval_policy=retrieval_policy,
    ),
    "clovanote_txt": run_meeting_workflow(
        sources["clovanote_txt"], domain_context,
        review_decision="approve",
    ),
    "audio_stt": run_meeting_workflow(
        sources["audio_stt"], domain_context,
        review_decision="edit",
        review_edits={
            "meeting_summary": "고객 문의 자동화 PoC의 범위·금지 행동·담당자별 후속 조치를 근거와 함께 정리했습니다."
        },
        transcriber=reviewed_fixture_stt,
    ),
}
workflow_result = {
    "coding_agent_brief": coding_agent_brief,
    "scenarios": {
        name: compact_workflow_result(result)
        for name, result in workflow_runs.items()
    },
    "full_records": {
        name: result["record"] for name, result in workflow_runs.items()
    },
}
expected_nodes = [
    "policy", "input_normalize", "stt_optional", "structure",
    "evidence", "human_review", "export_draft",
]
assert all(
    [event["node"] for event in item["trace"]] == expected_nodes
    for item in workflow_result["scenarios"].values()
)
assert all(item["status"] == "DRAFT_READY" for item in workflow_result["scenarios"].values())
assert all(item["external_write"] is False for item in workflow_result["scenarios"].values())
save_json("05_workflow_runs.json", workflow_result)
workflow_result["scenarios"]

## 6차시 · LLM Provider · Cost Guardrail

기본 `Run All`은 API와 CLI를 호출하지 않습니다. OpenAI는 `OPENAI_LIVE_OPT_IN=1`과 환경변수 key가 모두 있을 때만 선택하며, 모델 접근 불가를 fixture 성공으로 위장하지 않고 `MODEL_NOT_AVAILABLE`로 남깁니다.

In [ ]:
import os
from types import SimpleNamespace

provider_options = diagnose_provider_options()
cli_dry_runs = {
    name: run_optional_cli_prompt(name, "현재 회의 기록을 검토해 주세요.")
    for name in ("ollama", "codex", "claude_code")
}
RUN_OPENAI_LIVE = False
openai_result = run_optional_openai_record(
    envelopes["google_meet_text"], domain_context,
    env=os.environ if RUN_OPENAI_LIVE else {},
    model=os.getenv("OPENAI_MODEL", DEFAULT_OPENAI_MODEL),
    allow_fixture_fallback=True,
)

class LocalModelNotFound(Exception):
    status_code = 404

class FakeResponses:
    @staticmethod
    def create(**_kwargs):
        raise LocalModelNotFound("requested model does not exist")

model_boundary = run_optional_openai_prompt(
    "모델 가용성 경계 테스트",
    env={"OPENAI_LIVE_OPT_IN": "1"},
    client=SimpleNamespace(responses=FakeResponses()),
    model=DEFAULT_OPENAI_MODEL,
)
provider_result = {
    "options": provider_options,
    "cli_default_run_all": cli_dry_runs,
    "openai_default_run_all": openai_result,
    "model_not_available_boundary": model_boundary,
    "live_flags": {
        "openai": RUN_OPENAI_LIVE,
        "ollama": False, "codex_cli": False, "claude_code_cli": False,
    },
}
assert openai_result["provider_used"] == "fixture"
assert openai_result["fallback_reason"] == "OPENAI_LIVE_OPT_IN_REQUIRED"
assert openai_result["schema_valid"] is True
assert model_boundary["fallback_reason"] == "MODEL_NOT_AVAILABLE"
assert all(item["error_code"] == "CLI_LIVE_OPT_IN_REQUIRED" for item in cli_dry_runs.values())
assert all(
    item.get("command_executed", False) is False
    for item in provider_options.values()
    if isinstance(item, dict)
)
save_json("06_provider_diagnostics.json", provider_result)
provider_result

## 7차시 · LangGraph · Human Review

승인·수정·거절은 각각 다른 상태와 산출물을 만듭니다. 존재하지 않는 evidence ID는 사람이 승인하기 전 `HOLD`하며, 수정은 허용된 필드만 다시 검증합니다.

In [ ]:
review_inputs = {
    "approve": {},
    "edit": {
        "meeting_summary": "사람이 근거를 확인하고 배송 지연 기록 범위를 수정했습니다.",
        "todo_updates": {"0": {"owner": "민지", "due_date": "2026-09-05"}},
    },
    "reject": {},
}
interruptible_runs = {}
for decision, edits in review_inputs.items():
    review_graph = build_interruptible_meeting_graph()
    thread_id = f"day2-{decision}-review"
    started = start_interruptible_meeting_review(
        review_graph,
        sources["google_meet_text"],
        domain_context,
        thread_id=thread_id,
        retrieval_policy=retrieval_policy,
    )
    resumed = resume_interruptible_meeting_review(
        review_graph,
        thread_id=thread_id,
        decision=decision,
        edits=edits,
    )
    interruptible_runs[decision] = {
        "start": {
            "status": started["status"],
            "thread_id": started["thread_id"],
            "checkpointer": started["checkpointer"],
            "interrupt": started["interrupts"][0],
            "external_write": started["external_write"],
        },
        "resume": {
            "status": resumed["status"],
            "review": resumed["review"],
            "export_status": resumed["exports"]["status"],
            "trace": resumed["trace"],
            "external_write": resumed["external_write"],
        },
    }

boundary_record = MeetingRecord.model_validate(
    record_contract_result["actual_record"]
)
boundary_payload = boundary_record.model_dump(mode="json")
boundary_payload["todos"][0]["evidence_ids"] = ["s999"]
evidence_boundary = validate_record_evidence(
    MeetingRecord.model_validate(boundary_payload),
    envelopes["google_meet_text"],
)
human_review_result = {
    "graph": {
        "framework": "LangGraph",
        "checkpointer": "InMemorySaver",
        "pause": "interrupt()",
        "resume": "Command(resume=...)",
        "conditional_routes": [
            "evidence → human_review | evidence_hold",
            "human_review → export_draft | review_rejected",
        ],
    },
    "decisions": interruptible_runs,
    "unknown_evidence_boundary": evidence_boundary,
}
assert all(
    item["start"]["status"] == "WAITING_FOR_HUMAN_REVIEW"
    for item in interruptible_runs.values()
)
assert interruptible_runs["approve"]["resume"]["status"] == "DRAFT_READY"
assert interruptible_runs["edit"]["resume"]["status"] == "DRAFT_READY"
assert interruptible_runs["reject"]["resume"]["status"] == "REJECTED"
assert interruptible_runs["reject"]["resume"]["export_status"] == "SKIPPED_NOT_APPROVED"
assert all(
    item["resume"]["external_write"] is False
    for item in interruptible_runs.values()
)
assert evidence_boundary == ["TODO_1_UNKNOWN_EVIDENCE:s999"]
save_json("07_human_review.json", human_review_result)
human_review_result

## 8차시 · Desktop App Package

세 시나리오의 결과를 로컬 Markdown과 이메일 초안으로 만들고 같은 핵심 기능을 Desktop UI에서 실행합니다. 수신자는 비어 있고 발송은 `false`입니다. 소스 실행과 Docker 실행, macOS PKG·Windows EXE 전달 경로까지 확인한 뒤 Day 2와 기존 Day 1 회귀 Test를 실행합니다.

In [ ]:
markdown_files = {}
email_drafts = {}
for name, result in workflow_runs.items():
    record = MeetingRecord.model_validate(result["record"])
    markdown_text = result["exports"]["markdown"]
    markdown_path = save_text(f"08_{name}_meeting.md", markdown_text)
    markdown_files[name] = str(markdown_path.relative_to(ROOT))
    email_drafts[name] = render_email_draft(record, audience="internal")

save_json("08_email_drafts.json", email_drafts)
desktop_delivery = {
    "source_run": "cd desktop-app/meeting-intelligence && python -m uvicorn app.main:app --host 127.0.0.1 --port 8766",
    "docker_run": "cd desktop-app/meeting-intelligence && docker compose up --build",
    "browser": "http://127.0.0.1:8766",
    "windows_exe": "desktop-app/meeting-intelligence/dist/MeetingIntelligence-Windows.exe",
    "macos_pkg": "desktop-app/meeting-intelligence/dist/MeetingIntelligence-macOS.pkg",
    "human_review_required": True,
    "external_write": False,
}
focused_test = run_command(
    sys.executable, "-m", "pytest", "-q", "tests/test_day2_meeting_workflow.py"
)
day1_suite = run_command(
    sys.executable, "-m", "pytest", "-q",
    "tests/test_day1_agent.py",
    "tests/test_langchain_langgraph_lab.py",
    "tests/test_meeting_agent_workflow.py",
    "tests/test_openai_provider.py",
    "tests/test_ollama_tool_agent.py",
)
export_result = {
    "markdown_files": markdown_files,
    "email_drafts": email_drafts,
    "desktop_delivery": desktop_delivery,
    "checks": {
        "all_emails_unsent": all(item["send"] is False for item in email_drafts.values()),
        "all_external_write_false": all(item["external_write"] is False for item in email_drafts.values()),
        "focused_test_returncode": focused_test["returncode"],
        "day1_suite_returncode": day1_suite["returncode"],
    },
}
assert export_result["checks"] == {
    "all_emails_unsent": True,
    "all_external_write_false": True,
    "focused_test_returncode": 0,
    "day1_suite_returncode": 0,
}
save_json("08_export_drafts.json", export_result)
export_result

## 완료 확인

- Day 2의 1~8차시 결과 파일을 확인했습니다.
- 정상 경로와 가장 중요한 실패 경로를 모두 실행했습니다.
- 외부 서비스 저장·게시·발송과 자동 메일이 기본값 `false`임을 확인했습니다.
- Codex·Claude Code 결과는 test와 diff를 사람이 검토한 뒤에만 반영합니다.